# 02 — ObjectScale S3 Throttler
**Objective:** validate native S3 throttling — a noisy tenant is capped while a critical workload stays inside its SLA and recovers.

**Customer sees:** measured phases with PASS / FAIL / INCONCLUSIVE verdicts against approved thresholds, and a live TPS/latency chart.
**TC:** TC2.1–TC2.11

**Flow:** rules are verified and stored first (throttler still disabled) so they have fully propagated before enforcement is measured → baseline (critical only, paced) → contention (+noisy, throttler disabled) → simulation (dry-run) → enforcement (HTTP 429) → recovery → post-baseline (noisy stopped) → prefix rule → Spark write as the throttled tenant → restore.

> ObjectScale caches the rule set for up to `rules_config_cache_expiration_time` = 300 s (read-only). The rules are staged by `99_reset` minutes before this notebook runs, so the propagation wait is normally already over — a short countdown appears only if they are still younger than 300 s. A verdict the evidence cannot decide is INCONCLUSIVE — never a pass.

**Noisy host:** `NOISY_HOST` empty = noisy PUTs run locally on this host; `NOISY_HOST=192.168.1.41:8765` delegates only tenant-noisy control/events to `poc/noisy_agent.py` while critical GETs, phases, verdicts and the chart stay here. Remote mode refuses to run if the two host clocks differ by more than 50 ms, and re-checks the offset at every phase boundary.

## Step 1 — Session, tenant isolation, effective policy
Prove noisy/critical are distinct IAM principals, then **read** the current throttler state. The pre-demo backup must exist before anything is changed. Any leftover load from an interrupted run is stopped first.

In [ ]:
import atexit, json, os, time
import ipywidgets as w
import pandas as pd
from IPython.display import HTML, display
from poc import config, results, viz
from poc import throttle_eval as te
from poc.loadgen import LoadGen, RemoteNoisy
from poc.obs_admin import MgmtApi, Throttler
import poc.spark_helpers as sh

_old = globals().get("_POC_LG")
if _old is not None:
    _old.stop()
spark = sh.build_session("obs-poc", bucket_users={
    config.BUCKET_DATA: "admin", config.BUCKET_THROTTLE: "noisy"})
viz.table(pd.DataFrame(sh.versions(spark).items(), columns=["component", "version"]))

PHASE_S = int(os.environ.get("THROTTLE_PHASE_S", "60"))
SETTLE_S = int(os.environ.get("THROTTLE_SETTLE_S", "120"))
CRITICAL_RATE = float(config.CRITICAL_RATE or 0)
RULE_PROPAGATION_S = 300          # rules_config_cache_expiration_time (read-only)
DUR = {"baseline": PHASE_S, "contention": PHASE_S, "simulation": PHASE_S,
       "enforced": 2 * PHASE_S, "recovery": PHASE_S, "post-baseline": PHASE_S,
       "burst": max(15, PHASE_S // 3)}
RID = results.run_id()

lg = LoadGen(max_keys=0, noisy_chunk=config.NOISY_OBJECT_KB * 1024,
             noisy_random=bool(config.PAYLOAD_RANDOM),
             critical_objects=config.CRITICAL_OBJECTS,
             critical_object_kb=config.CRITICAL_OBJECT_KB)
remote = None
if config.NOISY_HOST:
    if not config.NOISY_AGENT_TOKEN:
        raise RuntimeError("NOISY_HOST is set but NOISY_AGENT_TOKEN is empty — "
                           "the remote agent requires the shared X-POC-Token header")
    remote = RemoteNoisy(lg, config.NOISY_HOST, token=config.NOISY_AGENT_TOKEN,
                         run_id=RID, max_keys=0,
                         noisy_random=bool(config.PAYLOAD_RANDOM))
    try:
        offset, h = remote.check_clock()
        s3ok = (h.get("s3") or {}).get("ok")
        if not s3ok:
            raise RuntimeError("noisy agent cannot reach ObjectScale S3 on 9021: "
                               f"{(h.get('s3') or {}).get('error')}")
        if remote.running("tenant-noisy"):
            remote.stop_noisy()          # leftover from an interrupted run
        st = remote.status()
        if (st.get("noisy") or {}).get("running"):
            raise RuntimeError("remote noisy agent still reports running after stop")
        iso_rows, iso_ok = remote.verify_tenants()
    except Exception:
        remote.close()
        raise
else:
    iso_rows, iso_ok = lg.verify_tenants()
_POC_LG = lg
viz.table(pd.DataFrame(iso_rows))
thr = Throttler(MgmtApi())
pol = te.effective_policy(thr)
viz.table(pd.DataFrame([{"mode": pol["mode"], "response_code": pol["response_code"],
                         "existing_rules": len(pol["rules"]),
                         "backup": str(config.BACKUP_FILE),
                         "backup_exists": config.BACKUP_FILE.exists()}]))
if not config.BACKUP_FILE.exists():
    raise RuntimeError(f"{config.BACKUP_FILE} is missing — run scripts/prep_obs.py "
                       "while the throttler is in its pre-demo state")
if pol["mode"] == "enabled":
    raise RuntimeError("the throttler is already enforcing — this demo must start "
                       "from mode disabled/dry-run (restore the pre-demo state first)")
print("run id:", RID, "| phase seconds:", DUR,
      "| CRITICAL_RATE:", CRITICAL_RATE,
      "| NOISY_THREADS:", config.NOISY_THREADS,
      "| NOISY_OBJECT_KB:", config.NOISY_OBJECT_KB,
      "| PAYLOAD_RANDOM:", config.PAYLOAD_RANDOM,
      "| CRITICAL_OBJECTS:", config.CRITICAL_OBJECTS,
      "x", config.CRITICAL_OBJECT_KB, "KiB")
if remote is not None:
    print("noisy mode: REMOTE", remote.base_url,
          f"| U01/U02 clock offset {offset:+.1f} ms | agent S3 ok={s3ok}")
else:
    print("noisy mode: LOCAL (NOISY_HOST is empty)")
viz.banner(iso_ok, "tenant isolation",
           "distinct principals; effective policy read (unchanged)")

out = w.Output(); display(out)
chart = viz.LiveChart(lg)
PHASE_AUX = {}

def finish_phase(name):
    """Close a measured phase, settle its tail events, re-check the remote
    clock and record contact stats. Returns the aux dict."""
    end = lg.end_phase()
    aux = {"end": end, "start": None, "settle_error": None,
           "clock_offset_ms": None, "clock_rtt_ms": None, "clock_error": None,
           "remote": None, "contact_ok": None, "max_outage_s": None}
    try:
        lg.settle(end, timeout_s=30)
    except Exception as e:
        aux["settle_error"] = str(e)
    win = lg.phase_window(name) or (None, end)
    aux["start"] = win[0]
    if remote is not None:
        try:
            m = remote.recheck_clock(timeout=10)
            aux["clock_offset_ms"] = m.get("offset_ms")
            aux["clock_rtt_ms"] = m.get("rtt_ms")
        except Exception as e:
            aux["clock_error"] = str(e)
        try:
            st = remote.stats()
            aux["remote"] = {k: st.get(k) for k in
                             ("events_pulled", "poll_errors", "max_gap_s",
                              "contact_age_s", "last_error", "event_gap")}
            aux["contact_ok"] = bool(st.get("contact_ok"))
            if win[0]:
                aux["max_outage_s"] = remote.max_outage_between(win[0], end)
        except Exception as e:
            aux["contact_ok"] = False
            aux["contact_error"] = str(e)
    PHASE_AUX[name] = aux
    return aux

def clock_ok(name):
    a = PHASE_AUX.get(name, {})
    if a.get("clock_error"):
        return False
    off = a.get("clock_offset_ms")
    return off is None or abs(off) <= 50

def contact_reason(name):
    """None when the phase is clean, else the FAIL reason."""
    if remote is None:
        a = PHASE_AUX.get(name, {})
        return ("settle failed: " + a["settle_error"]) if a.get("settle_error") else None
    a = PHASE_AUX.get(name, {})
    if a.get("settle_error"):
        return "settle failed: " + a["settle_error"]
    if a.get("contact_ok") is False:
        err = ((a.get("remote") or {}).get("last_error")
               or a.get("contact_error") or "poll failing")
        return f"lost contact with noisy host ({err})"
    if (a.get("remote") or {}).get("event_gap"):
        return ("noisy events dropped before pull: "
                + a["remote"]["event_gap"])
    if (a.get("max_outage_s") or 0) > 5:
        return f"lost contact with noisy host ({a['max_outage_s']:.1f}s poll gap)"
    if a.get("clock_error"):
        return "clock check failed: " + a["clock_error"]
    if not clock_ok(name):
        return f"clock offset {a.get('clock_offset_ms')} ms exceeds 50 ms"
    return None

def aux_metrics(name):
    a = PHASE_AUX.get(name, {})
    return {"clock_offset_ms": a.get("clock_offset_ms"),
            "clock_rtt_ms": a.get("clock_rtt_ms"),
            "settle_error": a.get("settle_error"),
            "remote_pull": a.get("remote"),
            "max_outage_s": a.get("max_outage_s"),
            "contact_ok": a.get("contact_ok")}

def crit_p95_buckets(bucket_s=10):
    """Critical-GET p95 per `bucket_s` seconds from the contention start
    through the post-baseline end — shows where in the run latency moves."""
    import poc.loadgen as _lg
    phs = {p["name"]: p for p in getattr(lg, "phases", [])}
    a, b = phs.get("contention"), phs.get("post-baseline")
    if not a or not b:
        return []
    t0, t1 = a["start"], b["end"]
    buckets = {}
    for ts, lat in lg.critical_get_series():
        if t0 <= ts <= t1:
            buckets.setdefault(int((ts - t0) // bucket_s), []).append(lat)
    return [{"offset_s": k * bucket_s, "n": len(v),
             "p95_ms": round(_lg.pctile(v, 95), 1)}
            for k, v in sorted(buckets.items())]

**Presenter notes** (hidden)

Shared namespace bucket, separate principals — that is what principal-scoped rules discriminate on. LoadGen talks to S3 with SDK retries OFF so every 429 is seen as it arrives. Critical GETs are open-loop paced at CRITICAL_RATE; missed slots make a phase INCONCLUSIVE. In remote mode this cell also proves the agent reaches ObjectScale on 9021, writes as tenant-noisy, stops any leftover noisy load, and that U01/U02 clocks agree within 50 ms.

## Step 2 — Lab rules: staged by 99_reset, verified here (TC2.2)
Two rules for the noisy tenant: all PUTs capped at 50 TPS, and a tighter 10 TPS cap on one key prefix (`demo-raw-covered/*`). `99_reset` already stored them with the throttler disabled; this step verifies them against the live schema and reads their age. If they are missing they are staged now and the propagation clock restarts.

In [ ]:
from poc import demo_rules
NOISY_RULE = demo_rules.noisy_put_rule()
PREFIX_RULE = demo_rules.prefix_put_rule()
LIMIT, PREFIX_LIMIT = (NOISY_RULE["limit"]["value"],
                       PREFIX_RULE["limit"]["value"])
COVERED, CONTROL = "demo-raw-covered", f"{RID}/curated"
STAGED_FILE = config.REPORT_DIR / "rules_staged_at"
checks = {r["id"]: te.verify_rule(thr, r) for r in (NOISY_RULE, PREFIX_RULE)}
viz.table(pd.DataFrame([{"rule": rid, "schema_ok": c["ok"],
                         "problems": "; ".join(c["problems"]) or "—"}
                        for rid, c in checks.items()]))
display(HTML("<pre>" + json.dumps([NOISY_RULE, PREFIX_RULE], indent=2) + "</pre>"))
rules_ok = all(c["ok"] for c in checks.values())
res = results.write("TC2.2", "rules verified against the live schema", rules_ok,
                    {"rules": {rid: c["problems"] for rid, c in checks.items()}})
viz.banner(res["status"], "TC2.2", "rule schema + actions verified vs live API")
if not rules_ok:
    raise RuntimeError("a rule failed verification — nothing was pushed")

atexit.register(thr.restore_logged)          # restore even if a later cell dies
stored = thr.get_rules()
by_id = {r.get("id"): r for r in stored}
missing = [r["id"] for r in (NOISY_RULE, PREFIX_RULE) if r["id"] not in by_id]
if missing:
    keep = [r for r in stored if not demo_rules.is_demo_rule(r)]
    thr.set_mode("disabled")
    thr.put_config(keep + [NOISY_RULE, PREFIX_RULE])
    staged_at = time.time()
    STAGED_FILE.write_text(str(staged_at))
    print(f"rules were NOT staged by 99_reset (missing {missing}) — staged now; "
          f"the propagation clock restarts")
else:
    try:
        staged_at = float(STAGED_FILE.read_text())
    except Exception:
        staged_at = time.time()
        print("rules_staged_at missing — treating rule age as 0 (conservative)")
PUSHED_AT = staged_at
RULE_AGE_S = round(time.time() - PUSHED_AT, 1)
print(f"rules stored {RULE_AGE_S:.0f}s ago (mode {thr.get_mode()}); "
      f"enforcement waits until they are {RULE_PROPAGATION_S}s old")

**Presenter notes** (hidden)

principals accept user:<name>, iam::user:<id>, or the full IAM URN. paths are <namespace>/<bucket>/<key-glob>; the wildcard spans '/'. The prefix rule is scoped per principal — scoping per 'path' would give every object key its own counter. Existing non-demo rules are kept in the pushed set.

## Step 3 — Phase 1: baseline (TC2.1)
Critical GETs alone at the paced CRITICAL_RATE — the reference TPS and P50/P95/P99. Rules are stored but the throttler is disabled.

In [ ]:
lg.start(noisy=False, critical=True, critical_threads=4,
         critical_rate=CRITICAL_RATE)
# lg.start() seeds the critical objects once (head-then-put, skipped when
# they already exist for this run); say what was seeded.
print(f"critical reads: {config.CRITICAL_OBJECTS} x "
      f"{config.CRITICAL_OBJECT_KB} KiB objects, seeded once before baseline")
lg.start_phase("baseline"); chart.mark("baseline")
chart.refresh(out, seconds=DUR["baseline"])
aux = finish_phase("baseline")
ps_all = lg.phase_stats()
bp = ps_all["baseline"].get("tenant-critical GET", {})
bad = contact_reason("baseline")
st1 = "PASS" if (bp.get("ok", 0) >= 50 and bp.get("errors", 0) == 0
                 and bad is None) else ("FAIL" if bad else "INCONCLUSIVE")
res = results.write("TC2.1", "baseline critical GETs", status=st1,
    detail=(bad or f"{bp.get('tps', 0):.1f} TPS, p95 {bp.get('p95', 0):.0f} ms, "
                     f"p99 {bp.get('p99', 0):.0f} ms"),
    metrics={k: round(bp.get(k, 0), 2) for k in
             ("n", "tps", "p50", "p95", "p99", "errors", "throttled",
              "tenant-critical GET_paced", "tenant-critical GET_missed")}
            | {"critical_rate": CRITICAL_RATE, **aux_metrics("baseline")})
viz.banner(st1, "TC2.1 baseline", res["detail"])

**Presenter notes** (hidden)

Critical workload alone — the reference everything else is compared with. Latency is measured from the scheduled send time, so client stalls are counted, not hidden.

## Step 4 — Phase 2: contention, throttler disabled (TC2.3)
Noisy PUTs are added on top of the critical GETs with the throttler disabled. This measures how much the noisy writer affects the critical reader, if at all.

In [ ]:
lg.start_phase("contention"); chart.mark("contention")
lg.add_noisy(threads=config.NOISY_THREADS)
chart.refresh(out, seconds=DUR["contention"])
aux = finish_phase("contention")
ps_all = lg.phase_stats()
c = ps_all["contention"]
bl = ps_all["baseline"].get("tenant-critical GET", {})
ct, cn = c.get("tenant-critical GET", {}), c.get("tenant-noisy PUT", {})
bad = contact_reason("contention")
if min(ct.get("n", 0), cn.get("n", 0)) < 50:
    st3, why3 = "FAIL", "too few requests in the contention phase"
elif cn.get("throttled", 0):
    st3, why3 = "FAIL", f"{cn['throttled']} throttle responses while the throttler is disabled"
elif bad:
    st3, why3 = "FAIL", bad
else:
    delta = ct.get("p95", 0) - bl.get("p95", 0)
    effect = ("no measurable impact" if delta < 1
              else f"+{delta:.0f} ms measured impact")
    st3, why3 = "PASS", (f"critical p95 {bl.get('p95', 0):.0f} -> "
                         f"{ct.get('p95', 0):.0f} ms ({delta:+.0f} ms) "
                         f"with {cn.get('tps', 0):.0f} noisy PUT/s — "
                         f"{effect}")
res = results.write("TC2.3", "contention (throttler disabled)", status=st3, detail=why3,
    metrics={"baseline_p95": round(bl.get("p95", 0), 1),
             "contention_p95": round(ct.get("p95", 0), 1),
             "contention_p99": round(ct.get("p99", 0), 1),
             "critical_tps": round(ct.get("tps", 0), 1),
             "noisy_ok_tps": round(cn.get("ok", 0) / max(1e-9, c["_window"]["seconds"]), 1),
             "noisy_throttled": cn.get("throttled", 0), "mode": thr.get_mode(),
             "client_cpu_pct": {"baseline": ps_all["baseline"]["_window"]["client_cpu_pct"],
                                "contention": c["_window"]["client_cpu_pct"]},
             "critical_missed_slots": c.get("tenant-critical GET_missed", 0),
             **aux_metrics("contention")})
viz.banner(st3, "TC2.3 contention", why3)

**Presenter notes** (hidden)

Noisy PUTs are added on top of the critical GETs with the throttler disabled — this measures how much the noisy writer affects the critical reader, if at all. If there is no impact, the story is fairness and quotas: the throttler caps the noisy tenant before it can become a problem. TC2.3 is a sanity check (both workloads ran, nothing was throttled) — it does not fail on 'no impact'. Caveat for this lab: in LOCAL mode both tenants are driven from one 2-vCPU client host, and a CPU-only disturbance on that host (no S3 traffic) degrades the critical GETs just as much — so the size of this effect cannot be attributed to ObjectScale alone (reports/environment.md, 'Client-side contention'). client_cpu_pct is recorded per phase. With NOISY_HOST set, the noisy PUTs run on the second host and this client-side term is removed — compare the same phases between the two modes.

## Step 5 — Phase 3: simulation mode, dry-run (TC2.10)
Mode **dry-run** (ObjectScale simulation mode): the server only counts rule matches. Both workloads continue — the client must see **zero** throttle responses.

In [ ]:
thr.set_mode("dryrun")
lg.start_phase("simulation"); chart.mark("simulation (dry-run)")
chart.refresh(out, seconds=DUR["simulation"])
aux = finish_phase("simulation")
sp = lg.phase_stats()["simulation"]
sn = sp.get("tenant-noisy PUT", {})
srv_sim = te.server_counters(thr)
bad = contact_reason("simulation")
if sn.get("n", 0) < 50:
    st10, why10 = "INCONCLUSIVE", "too few noisy requests in the simulation phase"
elif sn.get("throttled", 0):
    st10, why10 = "FAIL", f"{sn['throttled']} client throttle responses in dry-run"
elif bad:
    st10, why10 = "FAIL", bad
else:
    st10, why10 = "PASS", (f"0 throttle responses in {sn['n']} noisy requests "
                           f"({sn.get('errors', 0)} other errors)")
res = results.write("TC2.10", "simulation mode (dry-run)", status=st10, detail=why10,
    metrics={"client_throttle_responses": sn.get("throttled", 0),
             "client_other_errors": sn.get("errors", 0),
             "noisy_ok_tps": round(sn.get("ok", 0) / max(1e-9, sp["_window"]["seconds"]), 1),
             "mode": thr.get_mode(),
             "server_metrics_available": srv_sim["available"],
             **aux_metrics("simulation")})
viz.banner(st10, "TC2.10 simulation", why10)

**Presenter notes** (hidden)

Dry-run validates rule matching safely. Simulated-match counters are a server-side concept: they are probed and reported as unavailable when the build does not expose them — never invented.

## Step 6 — Phase 4: enforcement, HTTP 429 (TC2.4)
Mode **enabled**, response code 429; the noisy tenant keeps running as fast as it can. The switch is read back, the notebook waits until the new mode is visibly in effect (tagged `switching`), then measures the `enforced` phase with both workloads running.

In [ ]:
wait_end = PUSHED_AT + RULE_PROPAGATION_S
if (left := wait_end - time.time()) > 0:
    countdown = w.HTML(); display(countdown)
    while (l := wait_end - time.time()) > 0:
        countdown.value = (f"<b>rule propagation: {l:.0f}s left</b> "
                           f"(rules staged by 99_reset; noisy still running, dry-run)")
        chart.refresh(out, seconds=min(10, l))
    print("propagation wait complete")
RULE_AGE_S = round(time.time() - PUSHED_AT, 1)
thr.set_response_code(429); thr.set_mode("enabled")
time.sleep(2)                                   # let the config write settle
eff = {"mode": thr.get_mode(), "response_code": thr.get_response_code(),
       "rules": [r.get("id") for r in thr.get_rules()]}
assert eff["mode"] == "enabled" and eff["response_code"] == 429, eff
lg.start_phase("switching"); chart.mark("mode -> enabled")
t_eff = te.wait_effective(lg, LIMIT, timeout_s=SETTLE_S)
print("mode switch visible after", t_eff, "s" if t_eff is not None
      else f"— NOT visible within {SETTLE_S}s; measuring anyway")

lg.start_phase("enforced"); chart.mark("enforced (429)")
half = DUR["enforced"] // 2
chart.refresh(out, seconds=half)
sustained_mid = lg.sustained(10, LIMIT * 0.25)
chart.refresh(out, seconds=DUR["enforced"] - half)
sustained_end = lg.sustained(10, LIMIT * 0.25)
aux = finish_phase("enforced")

ps_all = lg.phase_stats()
ep, cp = ps_all["enforced"], ps_all["contention"]
en, ec = ep.get("tenant-noisy PUT", {}), ep.get("tenant-critical GET", {})
st4, why4, enforced, enf_mode = te.tc24_status(
    ps_all, LIMIT, sustained=sustained_mid and sustained_end)
bad = contact_reason("enforced")
if bad:
    st4, why4 = "FAIL", bad
sig = te.signals(ps_all, LIMIT)
res = results.write("TC2.4", "enforcement (HTTP 429)", status=st4, detail=why4,
    metrics={"noisy": {k: round(en.get(k, 0), 2) for k in
                       ("n", "ok", "throttled", "r429", "r503_slowdown", "r503_other",
                        "errors", "retries_ok", "retries_fail", "boundary",
                        "ok_p50", "ok_p95", "ok_p99", "fail_p50", "fail_p95")},
             "noisy_ok_tps_before": round(sig["ok_tps_before"], 1),
             "noisy_ok_tps_enforced": round(sig["ok_tps"], 1),
             "limit_tps": LIMIT, "codes": en.get("codes", {}),
             "critical_tps": round(ec.get("tps", 0), 1),
             "critical_p95": round(ec.get("p95", 0), 1),
             "critical_p99": round(ec.get("p99", 0), 1),
             "critical_throttled": ec.get("throttled", 0),
             "critical_errors": ec.get("errors", 0),
             "critical_missed_slots": ep.get("tenant-critical GET_missed", 0),
             "sustained_mid": sustained_mid, "sustained_end": sustained_end,
             "rule_age_at_enable_s": RULE_AGE_S, "seconds_to_effect": t_eff,
             "effective_config": eff,
             "enforcement_active": enforced, "enforcement_mode": enf_mode,
             **aux_metrics("enforced")})
viz.banner(st4, "TC2.4 enforcement", why4)

**Presenter notes** (hidden)

With client retries off ObjectScale shows both mechanisms: it queues requests to hold the accepted rate at the limit (latency tail grows) and rejects the overflow with 429 TooManyRequests. codes{} lists the S3 error codes exactly as received.

## Step 7 — Phase 5: sustained enforcement
**No configuration change here** — enforcement from Step 6 stays active and the noisy tenant keeps offering as fast as it can. This phase shows the cap holding for another minute: it measures critical latency while noisy is still capped at the limit. The verdict is written after post-baseline so the drift guard has both critical-alone readings. (Internal phase id stays `recovery` for result compatibility.)

In [ ]:
lg.start_phase("recovery"); chart.mark("sustained enforcement")
chart.refresh(out, seconds=DUR["recovery"])
aux = finish_phase("recovery")
print("sustained enforcement measured; verdict written after post-baseline")

**Presenter notes** (hidden)

Nothing changes in this phase — the already-running enforcement is observed for another minute to show the cap holds. The rule still admits ~50 noisy PUT/s by design, so the critical workload shares the system with that admitted load.

## Step 8 — Post-baseline + recovery verdict (TC2.5)
The noisy tenant is **stopped** while enforcement stays on: a second critical-alone reading that detects environment drift. TC2.5 is written here so `baseline_stable` and `recovery_improvement` have every phase they need.

In [ ]:
lg.stop_noisy()
lg.start_phase("post-baseline"); chart.mark("post-baseline (noisy stopped)")
chart.refresh(out, seconds=DUR["post-baseline"])
aux = finish_phase("post-baseline")
ps_all = lg.phase_stats()
verdicts = te.evaluate(ps_all, rule=NOISY_RULE)
REC_CHECKS = {"recovery_improvement", "baseline_stable",
              "critical_p95_within_sla", "paced_client_kept_up",
              "noisy_sustained_in_recovery", "critical_not_throttled"}
st5 = te.combine(verdicts["verdicts"], REC_CHECKS)
why5 = "; ".join(f"{v['name']}={v['verdict']}" for v in verdicts["verdicts"]
                 if v["name"] in REC_CHECKS)
contact_fails = {p: contact_reason(p) for p in
                 ("baseline", "contention", "simulation", "enforced",
                  "recovery", "post-baseline") if p in PHASE_AUX}
contact_fails = {p: r for p, r in contact_fails.items() if r}
if contact_fails:
    st5 = "FAIL"
    why5 = (why5 + "; " if why5 else "") + "; ".join(
        f"{p}: {r}" for p, r in contact_fails.items())
rp = ps_all.get("recovery", {})
rc, rn = rp.get("tenant-critical GET", {}), rp.get("tenant-noisy PUT", {})
rec_span = max(1e-9, rp.get("_window", {}).get("seconds", 1))
res = results.write("TC2.5", "sustained enforcement under throttling",
    status=st5, detail=why5,
    metrics={"baseline_p95": round(ps_all["baseline"].get("tenant-critical GET", {}).get("p95", 0), 1),
             "post_baseline_p95": round(ps_all.get("post-baseline", {}).get("tenant-critical GET", {}).get("p95", 0), 1),
             "contention_p95": round(ps_all["contention"].get("tenant-critical GET", {}).get("p95", 0), 1),
             "critical_p95": round(rc.get("p95", 0), 1),
             "critical_p99": round(rc.get("p99", 0), 1),
             "critical_tps": round(rc.get("tps", 0), 1),
             "critical_errors": rc.get("errors", 0),
             "critical_throttled": rc.get("throttled", 0),
             "noisy_offered_tps": round(rn.get("n", 0) / rec_span, 1),
             "noisy_accepted_tps": round(
                 (rn.get("n", 0) - rn.get("throttled", 0)) / rec_span, 1),
             "noisy_ok_tps": round(rn.get("ok", 0) / rec_span, 1),
             "noisy_throttled": rn.get("throttled", 0),
             "client_cpu_pct": {ph: ps_all[ph]["_window"]["client_cpu_pct"]
                                for ph in ("baseline", "contention", "simulation",
                                           "enforced", "recovery", "post-baseline")
                                if ph in ps_all},
             "checks": {v["name"]: v["detail"] for v in verdicts["verdicts"]
                        if v["name"] in REC_CHECKS},
             "phase_aux": {p: aux_metrics(p) for p in PHASE_AUX}})
pp = ps_all.get("post-baseline", {}).get("tenant-critical GET", {})
rc_p95 = rc.get("p95", 0)
v5 = next((v for v in verdicts["verdicts"]
           if v["name"] == "recovery_improvement"), {})
viz.banner(st5, "TC2.5 sustained enforcement",
           f"during sustained enforcement: critical p95 {rc_p95:.0f} ms "
           f"(baseline {res['metrics']['baseline_p95']:.0f}), noisy offered "
           f"{rn.get('n', 0) / rec_span:.0f} req/s, "
           f"{(rn.get('n', 0) - rn.get('throttled', 0)) / rec_span:.0f} "
           f"accepted · post-baseline (noisy stopped): critical p95 "
           f"{pp.get('p95', 0):.0f} ms"
           + (f" · {v5.get('mode')}" if v5.get("mode") else ""))

**Presenter notes** (hidden)

The mechanism is the already-running enforcement, observed longer — not a second action. recovery_improvement is the scored isolation verdict (mode 'improvement' when contention was measured, 'unaffected' when the critical workload never slowed); recovery_p95_vs_baseline is information only because single-host CPU noise made it unfair.

## Step 9 — Prefix-scoped rule (TC2.6)
The second rule caps the noisy tenant at 10 TPS under `demo-raw-covered/` only. The background noisy load is already stopped (Step 8), so these bursts run alone. Real traffic decides: one PUT burst to the covered prefix and one to a control prefix (`<run_id>/curated/`). PASS only if the covered prefix is held at its limit while the control prefix runs clearly above it.

In [ ]:
lg.start_phase("prefix-test"); chart.mark("prefix bursts")
cov = lg.noisy_burst(config.BUCKET_THROTTLE, COVERED, seconds=DUR["burst"], threads=4)
ctl = lg.noisy_burst(config.BUCKET_THROTTLE, CONTROL, seconds=DUR["burst"], threads=4)
pfx = te.evaluate_prefix(cov, ctl, PREFIX_LIMIT)
stored = [r.get("id") for r in thr.get_rules()]
if PREFIX_RULE["id"] not in stored:
    pfx = {"status": "FAIL", "reason": f"prefix rule not stored on the server: {stored}"}
cols = ("n", "ok", "ok_tps", "throttled", "r429", "errors", "ok_p95", "seconds")
viz.table(pd.DataFrame([{"prefix": "covered  " + COVERED, **{k: round(cov[k], 1) for k in cols}},
                        {"prefix": "control  " + CONTROL, **{k: round(ctl[k], 1) for k in cols}}]))
aux = finish_phase("prefix-test")
res = results.write("TC2.6", "prefix-scoped rule", status=pfx["status"], detail=pfx["reason"],
    metrics={"prefix_limit_tps": PREFIX_LIMIT, "tenant_limit_tps": LIMIT,
             "rule_path": PREFIX_RULE["filters"]["paths"]["values"][0],
             "covered": {k: round(cov[k], 1) for k in cols},
             "control": {k: round(ctl[k], 1) for k in cols},
             "covered_codes": cov["codes"], "control_codes": ctl["codes"],
             **aux_metrics("prefix-test")})
viz.banner(pfx["status"], "TC2.6 prefix rule", pfx["reason"])

**Presenter notes** (hidden)

Both bursts use the tenant-noisy identity with retries off. The control prefix is still subject to the tenant-wide rule, so it is expected near 50 TPS, the covered prefix near 10.

## Step 10 — Spark write as the throttled tenant (TC2.7)
The same Spark job runs while the noisy load keeps the tenant at its limit — in **one session**: S3A per-bucket credentials make `poc-spark-data` poc-admin and `poc-throttle` tenant-noisy (set in Step 1), so no session rebuilds are needed. A tiny untimed warm-up write loads the JVM classes and S3A, then the poc-admin / tenant-noisy pair runs `SPARK_WRITE_REPEATS` times (default 1) and the median is reported. Job size: `SPARK_WRITE_ROWS` (default 50,000) and `SPARK_WRITE_PARTITIONS` (default 4) — each output file costs several PUTs with the magic committer, so raise PARTITIONS for a bigger throttling effect (the step takes longer).

In [ ]:
import statistics
t_step = time.time()
lg.add_noisy(threads=config.NOISY_THREADS)
lg.start_phase("spark-write"); chart.mark("spark writes")

ROWS_W = config.SPARK_WRITE_ROWS
PARTS_W = config.SPARK_WRITE_PARTITIONS
REPEATS = max(1, config.SPARK_WRITE_REPEATS)
times, targets, t_start, write_errors = sh.benchmark_writes(
    spark, RID, [("admin", config.BUCKET_DATA),
                 ("noisy", config.BUCKET_THROTTLE)],
    ROWS_W, PARTS_W, repeats=REPEATS, warmup_bucket=config.BUCKET_DATA)
step_seconds = round(time.time() - t_step, 1)

t_admin = statistics.median(times["admin"]) if times["admin"] else None
t_noisy = statistics.median(times["noisy"]) if times["noisy"] else None
noisy_targets = [t for label, t in targets if label == "noisy"]
during = lg.snapshot(since=t_start).get("tenant-noisy", {}).get("PUT", {})
n429 = during.get("throttled", 0)
s3a_stats, s3a_throttled = {}, 0
for tgt in noisy_targets:
    st = sh.s3a_committer_stats(config.BUCKET_THROTTLE, tgt)
    s3a_stats[tgt] = st
    s3a_throttled += sum(v for k, v in st.items()
                         if "throttl" in k and isinstance(v, (int, float)))
mode_now = thr.get_mode()
bids = sh.bucket_identities(
    spark, [config.BUCKET_DATA, config.BUCKET_THROTTLE])
aux = finish_phase("spark-write")
if write_errors:
    e0 = write_errors[0]
    st7, why7 = "FAIL", (f"Spark write as {e0['label']} failed: "
                         f"{e0['error']}")
elif not times["noisy"] or not times["admin"]:
    st7, why7 = "FAIL", ("Spark write runs incomplete: "
                         f"admin={len(times['admin'])}, "
                         f"noisy={len(times['noisy'])}")
elif bids.get(config.BUCKET_THROTTLE) != "noisy":
    st7, why7 = "FAIL", (f"poc-throttle S3A identity is "
                         f"{bids.get(config.BUCKET_THROTTLE)}, not tenant-noisy")
elif mode_now != "enabled" or not (n429 or s3a_throttled):
    st7, why7 = "INCONCLUSIVE", (f"job completed in {t_noisy}s, but no throttling was "
                                 f"observed for tenant-noisy while it ran (mode={mode_now})")
elif contact_reason("spark-write"):
    st7, why7 = "FAIL", contact_reason("spark-write")
else:
    st7, why7 = "PASS", (f"median {t_noisy}s vs {t_admin}s as poc-admin "
                         f"({t_noisy / max(t_admin or 0.1, 0.1):.1f}x); "
                         f"{n429} tenant-noisy 429s during the job "
                         f"(load generator + Spark)")
res = results.write("TC2.7", "spark write as throttled tenant", status=st7, detail=why7,
    metrics={"seconds_admin_runs": times["admin"],
             "seconds_noisy_runs": times["noisy"],
             "seconds_admin_median": t_admin, "seconds_noisy_median": t_noisy,
             "slowdown": round(t_noisy / max(t_admin or 0.1, 0.1), 2)
                        if t_noisy and t_admin else None,
             "step_seconds": step_seconds,
             "spark_write_rows": ROWS_W, "spark_write_partitions": PARTS_W,
             "spark_write_repeats": REPEATS,
             "write_errors": write_errors or None,
             "bucket_identities": bids,
             "mode": mode_now,
             "tenant_noisy_429_during_job": n429,
             "tenant_noisy_ok_tps_during_job": round(
                 during.get("ok", 0) / max(1e-9, time.time() - t_start), 1),
             "s3a_committer_stats": s3a_stats,
             "s3a_throttled_requests": s3a_throttled,
             **aux_metrics("spark-write")})
viz.table(pd.DataFrame([
    {"identity": "poc-admin (not in the rule)", "runs_s": times["admin"],
     "median_s": t_admin},
    {"identity": "tenant-noisy (rule enforced)", "runs_s": times["noisy"],
     "median_s": t_noisy}]))
viz.banner(st7, "TC2.7 Spark under throttling", why7)

**Presenter notes** (hidden)

One session, two identities: fs.s3a.bucket.<bucket>.access.key credentials set in Step 1 make poc-spark-data poc-admin and poc-throttle tenant-noisy — verified per bucket by bucket_identities before the verdict. SPARK_WRITE_REPEATS>1 repeats the admin/noisy pair and reports the median. The warm-up is a tiny untimed write that only loads JVM classes and S3A. S3A retries throttled requests (fs.s3a.retry.throttle.*), so the job slows down instead of failing. The 429 count is tenant-noisy 429s during the job from the load generator plus Spark combined; Spark's own throttling is only claimed when the S3A committer counters show it (s3a_throttled_requests). Raise SPARK_WRITE_PARTITIONS for a bigger effect; the step takes longer.

## Step 11 — Stop the load: worker lifecycle (TC2.11)
Stop and restart the noisy group once to prove it restarts safely, then stop every worker. lg.stop() also does the final remote flush so no tail events are dropped.

In [ ]:
leftover = lg.stop_noisy()
restarted = lg.add_noisy(1) == 1
time.sleep(3)
restarted = restarted and lg.running("tenant-noisy")
leftover += lg.stop()
res = results.write("TC2.11", "worker lifecycle", restarted and leftover == 0,
                    {"noisy_restartable": restarted, "lingering_workers": leftover,
                     "settle_error": lg.last_settle_error})
viz.banner(res["status"], "TC2.11 worker lifecycle",
           "restart ok, no lingering workers" if res["passed"]
           else f"restartable={restarted}, lingering={leftover}")

**Presenter notes** (hidden)

stop() joins every worker; a worker that outlives its join timeout is kept in the count instead of being forgotten. In remote mode the same calls go to the agent on the noisy host and lg.stop() also closes the event poller.

## Step 12 — Verdicts + server-side correlation (TC2.9)
Score all phases against the approved thresholds and correlate the client-side throttle responses with ObjectScale's own counters where the build exposes them. The status **is** the overall verdict — INCONCLUSIVE is not a pass.

In [ ]:
ps_all = lg.phase_stats()
verdicts = te.evaluate(ps_all, rule=NOISY_RULE)
srv = te.server_counters(thr)
rows = [{"check": v["name"], "verdict": v["verdict"], "detail": v["detail"]}
        for v in verdicts["verdicts"]]
contact_fails = {p: contact_reason(p) for p in PHASE_AUX}
contact_fails = {p: r for p, r in contact_fails.items() if r}
if contact_fails:
    verdicts["overall"] = "FAIL"
    rows.append({"check": "remote_contact_or_clock", "verdict": "FAIL",
                 "detail": "; ".join(f"{p}: {r}" for p, r in contact_fails.items())})
display(pd.DataFrame(rows).style.apply(viz.status_style, column="verdict", axis=1))
corr = {"client_throttled_enforced_phase": ps_all.get("enforced", {}).get(
            "tenant-noisy PUT", {}).get("throttled", 0),
        "server_metrics_available": srv["available"], "probed": srv["probed"]}
if srv["available"]:
    corr["server_data"] = srv["data"]
else:
    print("server-side throttle counters are not exposed on this build — "
          "client-side observations only")
res = results.write("TC2.9", "verdicts + correlation", status=verdicts["overall"],
                    metrics={"overall": verdicts["overall"],
                             "enforcement_mode": verdicts["enforcement_mode"],
                             "thresholds": verdicts["thresholds"],
                             "verdicts": rows, **corr,
                             "clock_offsets": {p: a.get("clock_offset_ms")
                                               for p, a in PHASE_AUX.items()},
                             "contact_failures": contact_fails,
                             "critical_p95_10s": crit_p95_buckets(),
                             "post_baseline_p95": round(ps_all.get(
                                 "post-baseline", {}).get(
                                 "tenant-critical GET", {}).get("p95", 0), 1)})
viz.banner(verdicts["overall"], "TC2.9 overall verdict",
           f"enforcement mode: {verdicts['enforcement_mode']}")

**Presenter notes** (hidden)

A 50 TPS limit does not yield exactly 50 ok/s — the thresholds carry a stated tolerance. They are customer-approved values in te.DEFAULT_THRESHOLDS, not tuned to the result. recovery_improvement is the scored isolation verdict; recovery_p95_vs_baseline is information only.

## Step 13 — Phase summary
Every phase side by side: request counts, TPS, latencies, throttle responses, errors, paced slots and client CPU.

In [ ]:
def _win(gs):
    w = gs.get("_window")
    return w if isinstance(w, dict) else {}
summary = [{"phase": ph, "client_cpu_pct": _win(gs).get("client_cpu_pct"),
            "workload": k, "n": v.get("n"), "tps": round(v.get("tps", 0), 1),
            "ok_p50": round(v.get("ok_p50", 0)), "ok_p95": round(v.get("ok_p95", 0)),
            "ok_p99": round(v.get("ok_p99", 0)), "throttled": v.get("throttled", 0),
            "r429": v.get("r429", 0), "errors": v.get("errors", 0),
            "boundary": v.get("boundary", 0),
            "paced": gs.get("tenant-critical GET_paced"),
            "missed": gs.get("tenant-critical GET_missed")}
           for ph, gs in ps_all.items() for k, v in gs.items()
           if k != "_window" and isinstance(v, dict) and "n" in v]
viz.table(pd.DataFrame(summary))
chart.refresh(out, seconds=2)

**Presenter notes** (hidden)

client_cpu_pct = CPU utilisation of the load-generator host in that phase. Shaded bands mark the phases; the dotted line is the throttled (429) rate. `boundary` counts requests that were still in flight when their phase ended.

## Step 14 — Restore (TC2.8)
Restore the pre-demo throttler backup and verify the live state against the backup file — mode, response code and rule IDs.

In [ ]:
backup = json.loads(config.BACKUP_FILE.read_text())
want = {"mode": backup["mode"], "response_code": int(backup["response_code"]),
        "rules": sorted(r.get("id") for r in backup["config"].get("throttle", {}).get("rules", []))}
thr.restore()
live = {"mode": thr.get_mode(), "response_code": thr.get_response_code(),
        "rules": sorted(r.get("id") for r in thr.get_rules())}
res = results.write("TC2.8", "restore pre-demo throttler state", live == want,
                    {"live": live, "backup": want})
viz.banner(res["status"], "TC2.8 restore",
           f"mode={live['mode']}, response_code={live['response_code']}, "
           f"rules={live['rules'] or 'none'}")
spark.stop()

**Presenter notes** (hidden)

Backup: reports/throttle_backup.json — written once by scripts/prep_obs.py from a clean pre-demo state. The restore is also registered at kernel exit.